In [32]:
import os
import sys
import time
import warnings
from functools import partial

In [33]:
from abc import ABC, abstractmethod
from typing import Callable, Optional

In [34]:
import gymnasium as gym
import ale_py
from gymnasium.wrappers import AtariPreprocessing, FrameStackObservation
from stable_baselines3.common.atari_wrappers import FireResetEnv, EpisodicLifeEnv, ClipRewardEnv

In [35]:
warnings.filterwarnings("ignore")
gym.register_envs(ale_py)
if os.path.abspath("package") not in sys.path: sys.path.append(os.path.abspath("package"))

In [36]:
from package.environment import GymPreprocessing, create_breakout_env_gym
from package.dqn_types import ModelParameters, PathsParameters
from package.Logger import SmartLogger, LogsConfig

In [37]:
from stable_baselines3 import PPO
from stable_baselines3.common.base_class import BaseAlgorithm
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.evaluation import evaluate_policy

In [38]:
def is_notebook() -> bool:
    try:
        import IPython
        shell = IPython.get_ipython().__class__.__name__
        if shell == "ZMQInteractiveShell":
            return True
        elif shell == "TerminalInteractiveShell":
            return False
        else:
            return False
    except (NameError, ImportError):
        return False

In [86]:
class Support(ABC):
    @abstractmethod
    def freq(self) -> int:
        pass

    @abstractmethod
    def __call__(self) -> dict[str, int | float | bool | str]:
        pass

In [108]:
class EvaluateReward(Support):
    def __init__(
            self,
            network: BaseAlgorithm,
            env: gym.Env,
            freq: int = 1,
            n_episodes: int = 10,
            deterministic: bool = False
    ):
        self.network: BaseAlgorithm = network
        self.env: gym.Env = env
        self._freq: int = freq
        self.n_episodes: int = n_episodes
        self.deterministic: bool = deterministic

    def freq(self) -> int:
        return self._freq

    def __call__(self) -> dict[str, int | float]:
        mean_reward, std_reward = evaluate_policy(self.network, self.env, self.n_episodes, self.deterministic)
        return dict(mean_reward=mean_reward, std_reward=std_reward)

In [109]:
class Checkpointer(Support):
    def __init__(self, network: BaseAlgorithm, freq: int = 1, path: str = "./models"):
        self.network: BaseAlgorithm = network
        self.path: str = path
        self._freq: int = freq

    def freq(self) -> int:
        return self._freq

    def __call__(self) -> dict[str, str | bool]:
        path: str = os.path.join(self.path, f"model_{time.time()}")
        self.network.save(path)
        return dict(checkpoint=path, succses=True)

In [110]:
class VideoWriter(Support):
    def __init__(self, network: BaseAlgorithm, freq: int = 1, path: str = "./models"):
        self.network: BaseAlgorithm = network
        self.path: str = path
        self._freq: int = freq

    def freq(self) -> int:
        return self._freq

    def __call__(self) -> dict[str, str | bool]:
        pass

In [113]:
class Callback(BaseCallback):
    def __init__(
            self,
            *assistants: Support,
            stop_criterion: Optional[Support] = None,
            writer: Optional[SmartLogger] = None,
            verbose: int = 0
    ):
        super().__init__(verbose)
        self.assistants: tuple[Support] = assistants
        self.stop_criterion: Optional[Support] = stop_criterion
        self.writer: Optional[SmartLogger] = writer
        self.initial_state: bool = True

    def _on_step(self) -> bool:
        # ----------------------------
        if self.initial_state:
            self.initial_state = False
            return True
        # ----------------------------
        for assistant in self.assistants:
            if self.n_calls % assistant.freq() != 0: continue
            for key, value in assistant().items():
                self.logger.record(key, value)
        # ----------------------------
        if self.writer is not None:
            if self.n_calls % self.writer.options.metrics_save_freq == 0:
                for key, value in self.logger.name_to_value.items():
                    if isinstance(key, str): key = key.replace("/", "_")
                    self.writer.set_scalar(time.time(), key, value)
        # ----------------------------
        if self.stop_criterion is not None:
            criterion: dict[str, int | float] = self.stop_criterion()
            if all(criterion.values()):
                return False
        return True

In [114]:
env_prep = GymPreprocessing(
    partial(AtariPreprocessing, noop_max=20, frame_skip=4, screen_size=64),
    partial(EpisodicLifeEnv),
    partial(FireResetEnv),
    partial(ClipRewardEnv),
    partial(FrameStackObservation, stack_size=4)
)

In [115]:
build_env: Callable[[], gym.Env] = lambda: create_breakout_env_gym(transform=env_prep)
envir: DummyVecEnv = DummyVecEnv(list(build_env for _ in range(8)))

In [116]:
model = PPO(
    "CnnPolicy",
    envir,
    verbose=0,
    learning_rate=1e-4,
    n_steps=1,
    batch_size=32,
    n_epochs=10,
    device="mps",
)

In [117]:
model_space: ModelParameters = ModelParameters()

In [118]:
paths_space = PathsParameters(exp_name="ppo", log_dir="breakout_logs")
logs_config = LogsConfig(paths_space.log_dir, metrics_save_freq=1, weights_save_freq=1, videos_save_freq=1)
logger = SmartLogger(model.__class__.__name__, options=logs_config, exp_name=paths_space.exp_name)

In [119]:
services: tuple[Support, ...] = (
    EvaluateReward(model, envir, freq=logger.options.weights_save_freq, n_episodes=10),
    Checkpointer(model, freq=logger.options.weights_save_freq, path=logger.model_paths[model.__class__.__name__])
)

In [120]:
last_upd: Optional[str] = logger.get_last_update(model.__class__.__name__)
model.load(last_upd) if last_upd else None

In [130]:
callback = Callback(*services, writer=logger)
model = model.learn(total_timesteps=8 * 1 * 10, callback=callback)

In [132]:
# logger.draw_scalars()

In [19]:
def make_env(seed=0):
    def _init():
        env = create_breakout_env_gym(transform=env_prep)
        return env

    return _init

In [35]:
make_atari_env("BreakoutNoFrameskip-v4", n_envs=8, seed=0).reset().shape

(8, 84, 84, 1)

In [42]:
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack


def train():
    # 1. Создаем окружение с базовыми обертками Atari (NoopReset, MaxAndSkip и т.д.)
    # n_envs=8 позволяет обучать агента в 8 окнах одновременно (сильно ускоряет процесс)
    env = make_atari_env("BreakoutNoFrameskip-v4", n_envs=8, seed=0)
    print(env.reset().shape)

    # 2. Склеиваем 4 кадра для понимания динамики движения
    env = VecFrameStack(env, n_stack=4)
    print(env.reset().shape)

    # 3. Создаем модель
    # CnnPolicy — специальная нейросеть для обработки изображений
    model = PPO(
        "CnnPolicy",
        env,
        verbose=1,
        learning_rate=1e-4,
        batch_size=256,
        n_steps=128,  # количество шагов до обновления градиента
        device="cuda"  # используй "cpu", если нет видеокарты NVIDIA
    )

    # 4. Запускаем обучение (начни с 1 000 000 шагов для видимого результата)
    print("Начинаю обучение...")
    model.learn(total_timesteps=10)

    # 5. Сохраняем модель
    model.save("ppo_breakout_model")
    print("Модель сохранена.")

In [43]:
train()

(8, 84, 84, 1)
(8, 84, 84, 4)
Using cpu device
Wrapping the env in a VecTransposeImage.
Начинаю обучение...
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 519      |
|    ep_rew_mean     | 0        |
| time/              |          |
|    fps             | 1552     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 1024     |
---------------------------------
Модель сохранена.


In [ ]:


def train():
    # 1. Создаем функцию для генерации одиночной среды
    def make_env(seed=0):
        def _init():
            env = create_breakout_env_gym(transform=env_prep)
            return env

        return _init

    # 2. Создаем параллельные среды (например, 8 штук)
    n_envs = 8
    env = SubprocVecEnv([make_env(seed=i) for i in range(n_envs)])
    # Установка seed для каждой среды через метод seed (или в reset в более новых версиях)
    env.seed(0)

    # 3. Склеиваем 4 кадра для понимания динамики движения
    # Примечание: env_prep уже содержит FrameStackObservation, 
    # поэтому VecFrameStack может быть избыточен, если в env_prep уже стоит stack_size=4.
    # Но для демонстрации совместимости оставляем.
    # env = VecFrameStack(env, n_stack=4) 

    # 4. Создаем модель
    model = PPO(
        "CnnPolicy",
        env,
        verbose=1,
        learning_rate=1e-4,
        batch_size=256,
        n_steps=128,
        device="cuda"
    )

    # 5. Запускаем обучение
    print("Начинаю обучение...")
    model.learn(total_timesteps=1_000_000)

    # 6. Сохраняем модель
    model.save("ppo_breakout_model")
    print("Модель сохранена.")


if __name__ == "__main__":
    train()